# Environment setup

Create the notebook environment from the `lakehouse_data` directory:

```bash
conda env create -f environment.yml
conda activate local-lakehouse
```

Select the `local-lakehouse` kernel in VS Code. From the repository root, start
Spark Connect, Hive Metastore, and MinIO:

```bash
docker compose up -d --build
```

The Spark Connect server provides Delta, Hadoop S3A, and MinIO credentials. The
notebook only needs the matching Spark Connect Python client.

# Start a spark Session with a catalog

In [1]:
from datetime import datetime, timezone

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .remote("sc://localhost:15002")
    .appName("notebook-overview")
    .getOrCreate()
)

spark.sql('SHOW DATABASES').show()
spark.sql('SHOW TABLES IN crawler').show()


+----------------+
|       namespace|
+----------------+
|catalog_examples|
|         crawler|
|         default|
|notebook_connect|
+----------------+

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|  crawler|    books_b0|      false|
|  crawler|responses_b0|      false|
|  crawler|responses_s0|      false|
+---------+------------+-----------+



In [5]:
spark.sql("SELECT * FROM crawler.books_b0 WHERE filename LIKE 'Manufacturing%'").show()

+--------------------+--------------------+--------------------+--------------------+--------------------+----------------+---------+--------------+--------------------+--------------------+-----------------+--------------------+
|          chapter_id|           file_hash|            filename|               title|              author|publication_date|part_name|chapter_number|        chapter_name|                text|   split_strategy|         ingested_at|
+--------------------+--------------------+--------------------+--------------------+--------------------+----------------+---------+--------------+--------------------+--------------------+-----------------+--------------------+
|366a409c53270a4d2...|66181e77605c584d2...|Manufacturing Con...|Manufacturing Con...|Edward S. Herman ...|            NULL|     NULL|            15|               Notes|### INTRODUCTION ...|table_of_contents|2026-10-06 18:08:...|
|7b06bc87881c6a60b...|66181e77605c584d2...|Manufacturing Con...|Manufacturing Co

In [4]:
spark.stop()